[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordCDT2026ClinicalUtility/blob/main/extension/solutions/02_comparing_models.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("clinical_utility") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/BradSegal/OxfordCDT2026ClinicalUtility.git"], check=True)
    importlib.invalidate_caches()

import plotly.io as pio

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    pio.renderers.default = "colab"
except ImportError:
    pio.renderers.default = "plotly_mimetype"
print("Course helpers ready.")

# Comparing models: reclassification and net-benefit differences

Allow about 30 minutes. Which recordings change decision when you switch from one
model to the other, and is that change worth making? In this section we will compute
the reclassification indices that papers report, put a paired uncertainty interval
on the difference in net benefit, audit the recordings that change decision, and
finally fix a model comparison that erroneously valued the two models at different
preferences.

The action is **send this recording for AF review**. Let's assume it is priced in
reviews as we've defined previously in the core notebook, i.e. $C_{TP}$ = 1,
$C_{FP}$ = 1, $C_{FN}$ = 5 and $C_{TN}$ = 0. Attempt this section on your own.
Replace each `...` and run the checker below it to see if you've achieved the
desired result. If you get stuck for about 10 minutes, consider using the stuck
switch below the problem (where there is one) or opening the worked copy in
`extension/solutions/`. Please note that a later cell that needs an unfinished
problem prints `Waiting for ...`: once that problem is done or switched, click in
the waiting cell, choose **Runtime > Run before**, then run the waiting cell.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

from clinical_utility import comparison, data, reference
from clinical_utility import extension_checks as check

scores = data.load_scores()
development = scores[scores["split"] == "development"].reset_index(drop=True)
y = development["af"].to_numpy()
old, new = development["random_forest"].to_numpy(), development["resnet"].to_numpy()
cut = 0.2

## Start with a change in action

Let's compare the random forest (old) and the raw ResNet (new) on the same
development recordings, with both reviewing at a cut of 0.2. The reclassification
table below, built with `pd.crosstab`, has the old action in its rows and the new
action in its columns, separately for recordings with and without AF. The
off-diagonal cells show where the decision changes.

In [ ]:
comparison.reclassification(development, "random_forest", "resnet", cut)

If the recording truly has AF, it is beneficial if it moves up into review and
harmful if it moves down. For recordings without AF the inverse is true. We can
quantify this with the net reclassification improvement (NRI), which summarises
these moves in two components, each a proportion within its own outcome group, and
adds them:

`event = P(up | AF) - P(down | AF)` and
`non_event = P(down | no AF) - P(up | no AF)`.

On development, we can see that 74 of 341 AF recordings move up and 9 move down; 639
of 3,370 recordings without AF move up and 110 move down. Adding the two components
equally weights each group's proportion, irrespective of the group sizes or the
relative cost of a miss and a false referral.

### Problem A: Make

Implement the `categorical_nri(old, new, y, cut)` function. It should output a
dictionary with the keys `event`, `non_event` and `total`. Build the old and new
review masks with `>=`, compare them, then take the group means. Pay careful
attention to the denominators: recordings that stayed put belong in them. Note that
there is no standard Python library that computes NRI, so we have to implement it
ourselves!

In [ ]:
def categorical_nri(old, new, y, cut):
    y = np.asarray(y, dtype=bool)
    before, after = np.asarray(old) >= cut, np.asarray(new) >= cut
    up, down = after & ~before, before & ~after
    event = up[y].mean() - down[y].mean()
    non_event = down[~y].mean() - up[~y].mean()
    return {"event": event, "non_event": non_event, "total": event + non_event}
_ = check.run("cmp_nri", categorical_nri)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# categorical_nri = check.use_reference("categorical_nri")

In [ ]:
if check.ready(categorical_nri=categorical_nri):
    nri = categorical_nri(old, new, y, cut)
    display(pd.DataFrame({
        "categorical NRI at the cut": nri,
        "continuous NRI": comparison.continuous_nri(old, new, y),
        "IDI": comparison.idi(old, new, y)}).round(3))

We can see here that the categorical components are +0.191 and -0.157, totalling
+0.034, which makes the ResNet look like an improvement over the random forest. Two
cut-free indices that reviewers commonly request in addition to this, shown beside
it, are the continuous NRI and the IDI. **Continuous NRI** considers any rise or
fall in score as a move, irrespective of how small; ties count neither way. **IDI**,
the integrated discrimination improvement, is the sum of the mean rise in score
among AF recordings and the mean fall among the rest. Here they are +0.985 and
+0.197, larger still.

The problem with both of these is that they can reward changes in score that do not
impact decision-making. For example, a continuous NRI move could just be a slight
increase in score for a recording that still remains far below the decision cutoff
and as a result does not change any decision; in addition, the IDI metric is
sensitive to the scale of probability, so if you make the model's predictions more
extreme, i.e., pushing up the scores for AF recordings and down for all others
without changing the relative order of scores, the IDI value will increase despite
not improving the performance of any cutoff value (in contrast, if you add a
constant value to all scores, the IDI value will not change). These should be
reported as descriptions of how the scores have changed; however, the overall
assessment of the 'better' model should be based on the net benefit when accounting
for the preference of the service in question.

## Put a preference and uncertainty on the comparison

The difference in net benefit uses our preference, the threshold probability $p_t$,
to determine how much we value each change of decision: a recording's change
contributes **new minus old** value to net benefit. We can calculate the difference
in net benefit by simply taking the average of those contributions across all
recordings. At $p_t$ = 0.2 a false referral weighs 0.25:

| A recording's change of decision | Recordings | Contribution each |
|---|---:|---:|
| AF, moves up into review | 74 | `+1` |
| AF, moves down out of review | 9 | `-1` |
| No AF, moves up into review | 639 | −0.25 |
| No AF, moves down out of review | 110 | +0.25 |

These can be summed and then divided by the 3,711 development recordings to produce
a difference in net benefit of -0.0181, demonstrating that the ResNet is worse at
this preference despite its positive NRI.

### Problem B: Make

Implement the `delta_net_benefit(y, old_review, new_review, p_t)` function. It
should define an inner function `statistic(y, old_review, new_review, axis=-1)` that
returns the mean contribution along `axis`. This should then be used to bootstrap
the difference by calling `stats.bootstrap((y, old_review, new_review), statistic,
paired=True, vectorized=True, n_resamples=2000, method="percentile",
rng=np.random.default_rng(20261106))`. `paired=True` resamples recordings with both
decisions and the label together. The function should output a dictionary with the
keys `estimate`, `low` and `high` (the 95% percentile interval).

In [ ]:
def delta_net_benefit(y, old_review, new_review, p_t):
    weight = p_t / (1 - p_t)

    def statistic(y, old_review, new_review, axis=-1):
        value = np.where(y, 1.0, -weight)
        return np.mean((new_review.astype(float) - old_review) * value, axis=axis)

    data_ = (np.asarray(y, dtype=bool), np.asarray(old_review, dtype=bool),
             np.asarray(new_review, dtype=bool))
    result = stats.bootstrap(data_, statistic, paired=True, vectorized=True, n_resamples=2000,
                             method="percentile", rng=np.random.default_rng(20261106))
    return {"estimate": float(statistic(*data_)), "low": float(result.confidence_interval.low),
            "high": float(result.confidence_interval.high)}
_ = check.run("cmp_bootstrap", delta_net_benefit)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# delta_net_benefit = check.use_reference("delta_net_benefit")

In [ ]:
if check.ready(delta_net_benefit=delta_net_benefit):
    display(pd.DataFrame({p_t: delta_net_benefit(y, old >= p_t, new >= p_t, p_t)
                          for p_t in (0.1, 0.2)}).round(4))

The results indicate that for a preference of 0.2, the difference in net benefit is
-0.0181 with a 95% interval of -0.0241 to -0.0124; the entire interval is below
zero. If we were to use a preference of $p_t$ = 0.1, which would represent a service
that values a miss more (a miss priced at roughly ten reviews), we would see a
difference of -0.0031 (-0.0073 to 0.0013). This interval includes zero, and so at
that preference these recordings cannot tell the two models apart. It is important
to note that the interval only describes sampling uncertainty for independent
recordings with both policies fixed. It does not consider the uncertainty from
fitting the models, from choosing the prices or from moving to a new service, and
several recordings from one patient would need the patient as the resampling unit.

## Audit the changed decisions

### Problem C: Make

Construct a function `changed_decisions(frame, old, new, cut,
p_t)`, where `old` and `new` are the names of the two score columns. The function
should return a DataFrame of only the recordings whose review decision changes, in
the same order as the input DataFrame, with the columns `recording`,
`reference_label`, the two score columns, `direction` (`"up"` or `"down"`) and
`contribution` (`+1` or `-1` for an AF recording moving up or down, −odds or +odds
for one without AF, where odds = `p_t / (1 - p_t)`).

In [ ]:
def changed_decisions(frame, old, new, cut, p_t):
    before, after = frame[old] >= cut, frame[new] >= cut
    changed = frame[before != after]
    moved_up = after[before != after]
    value = np.where(changed["af"], 1.0, -p_t / (1 - p_t))
    return pd.DataFrame({"recording": changed["recording"],
                         "reference_label": changed["reference_label"],
                         old: changed[old], new: changed[new],
                         "direction": np.where(moved_up, "up", "down"),
                         "contribution": np.where(moved_up, value, -value)}).reset_index(drop=True)
_ = check.run("cmp_changed", changed_decisions)

In [ ]:
# Stuck for about 10 minutes? Delete the # on the next line and run this cell.
# changed_decisions = check.use_reference("changed_decisions")

In [ ]:
if check.ready(changed_decisions=changed_decisions):
    audit = changed_decisions(development, "random_forest", "resnet", cut, cut)
    summary = audit.assign(group=np.where(audit["reference_label"] == "A", "AF", "no AF")).groupby(
        ["group", "direction"]).size().rename("recordings").to_frame()
    summary["per 1,000"] = 1000 * summary["recordings"] / len(development)
    display(summary.round(1))
    print("Sum of contributions / all recordings:", round(audit["contribution"].sum() / len(development), 4))
    display(audit.groupby(["reference_label", "direction"]).head(1))

Read the summary as consequences on the development split, in counts and per 1,000
recordings. Overall, we can see that switching to the ResNet would review 74 AF
recordings that the forest missed and lose 9, at the price of 639 new reviews of
recordings without AF, while 110 recordings without AF that the forest reviewed
would no longer be. Most of those are O recordings, which the ResNet scores as
'AF-like'. An example recording of each kind, with both scores, has been included
above, but scores and labels can only support that account; they cannot show a
waveform mechanism.

### Problem D: Fix the bug

Your colleague has written the function below to compare the net benefit of the two
models. They call it with each model's minimum-loss cutoff (forest 0.239, raw ResNet
0.800), and it reports that the ResNet does worse than reviewing no one. Can you fix
the bug in `compare_net_benefit` in place?

In [ ]:
def compare_net_benefit(frame, cutoffs, p_t):
    """Net benefit of each model's decisions at its cutoff, valued at the service's preference p_t."""
    y = frame["af"].to_numpy()
    result = {}
    for model, model_cut in cutoffs.items():
        review = frame[model].to_numpy() >= model_cut
        tp, fp = np.sum(review & y), np.sum(review & ~y)
        weight = p_t / (1 - p_t)
        result[model] = (tp - fp * weight) / len(y)
    return result


cutoffs = {model: float(reference.cheapest(reference.operating_points(
    development["af"], development[model], reference.DEFAULT_COSTS))["threshold"])
    for model in ("random_forest", "resnet")}
_ = check.show(compare_net_benefit, development, cutoffs, 0.2)
_ = check.run("cmp_preference", compare_net_benefit)

#### Cutoff and preference are different things

The problem is that the cutoff and preference are not the same thing. The cutoff is
the value we use to determine who should be reviewed; however, the preference $p_t$
is derived from the prices and represents the relative weight of incurring a false
referral. For a perfectly calibrated model, the cutoff that minimises the expected
loss is the same as the preference (if the cutoff is chosen on a sample, as with the
forest model's value of 0.239, there will still be a slight deviation), and so it is
an easy mistake to make. The ResNet model is poorly calibrated and so the cutoff
that minimises loss is 0.800; however, if we use this as our preference $p_t$, we
weight each false referral by 4.01 instead of 0.25. This is what caused the net
benefit of the ResNet model to be -0.337 as opposed to 0.053. The correct way to
compare the two models would be to calculate the net benefit given the same
preference value for both models, i.e., 0.060 for the forest model and 0.053 for the
ResNet model.

## Check your report

- What is the denominator for each component of the NRI?
- How is it possible to have a positive NRI but a negative difference in net
  benefit?
- Report the preference, the difference in net benefit, its interval, as well as the
  resampling unit.
- Describe one changed decision based on the recording ID, both models' scores, and
  the reference label.

**Question to consider:** if a service cared much more about misses, what would we
have to recalculate if we wanted to make a recommendation about which model to use?